# 01 Data Preparation

In [1]:
# Load libraries, functions, and raw data

import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve() / 'src'))

from data_processing import (
    load_reviews_json,
    build_reviews,
    build_authors,
    attach_author_id,
    drop_missing_reviews,
    pct,
    dedupe_reviews_by_content,
    build_hotels_agg,
    write_db,
    make_reproducible_samples
)

df_raw = load_reviews_json('../data/review.json')
df_raw.head()


,ratings,title,text,author,date_stayed,offering_id,num_helpful_votes,date,id,via_mobile
0,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...","“Truly is ""Jewel of the Upper Wets Side""”",Stayed in a king suite for 11 nights and yes i...,"{'username': 'Papa_Panda', 'num_cities': 22, '...",December 2012,93338,0,2012-12-17,147643103,False
1,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...",“My home away from home!”,"On every visit to NYC, the Hotel Beacon is the...","{'username': 'Maureen V', 'num_reviews': 2, 'n...",December 2012,93338,0,2012-12-17,147639004,False
2,"{'service': 4.0, 'cleanliness': 5.0, 'overall'...",“Great Stay”,This is a great property in Midtown. We two di...,"{'username': 'vuguru', 'num_cities': 12, 'num_...",December 2012,1762573,0,2012-12-18,147697954,False
3,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...",“Modern Convenience”,The Andaz is a nice hotel in a central locatio...,"{'username': 'Hotel-Designer', 'num_cities': 5...",August 2012,1762573,0,2012-12-17,147625723,False
4,"{'service': 4.0, 'cleanliness': 5.0, 'overall'...",“Its the best of the Andaz Brand in the US....”,I have stayed at each of the US Andaz properti...,"{'username': 'JamesE339', 'num_cities': 34, 'n...",December 2012,1762573,0,2012-12-17,147612823,False


In [2]:
# Rename 'date' column to 'review_date' for clarity

df_raw = df_raw.rename(columns={'date': 'review_date'})

## Flatten Ratings column

We see that the ratings column is nested and contains multiple rating aspects. To make the data accessibility easier, we flatten these ratings and merge them into the review table.

In [3]:
flattened_review_df = build_reviews(df_raw)
sorted_review_df = flattened_review_df.sort_values(by='review_date', ascending=False)
sorted_review_df

,title,text,author,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating
384302,“Place to stay in Downtown LA”,This is my first time staying in downtown when...,"{'username': 'netsurfe', 'num_cities': 3, 'num...",November 2012,77804,0,2012-12-20,147786903,False,4.0,4.0,5.0,5.0,5.0,5.0,4.0,NaN,NaN
266659,“Michigan Avenue/Grant Park”,"Great location, right on Michigan Ave. Bus sto...","{'username': 'janiejake', 'num_cities': 10, 'n...",August 2012,87573,0,2012-12-20,147767791,False,4.0,3.0,3.0,3.0,4.0,4.0,3.0,NaN,NaN
840520,“Excellent location”,Location: The location was very good as it was...,"{'username': 'P T', 'num_cities': 8, 'num_help...",August 2012,84137,0,2012-12-20,147777249,False,4.0,4.0,4.0,5.0,5.0,5.0,3.0,NaN,NaN
840518,"“Convenient location, but need a makeover”","First trip to Washington, children want to see...","{'username': 'INNALOO', 'num_cities': 17, 'num...",November 2012,84067,0,2012-12-20,147771855,False,5.0,5.0,4.0,4.0,5.0,3.0,3.0,NaN,NaN
781827,“four seasons for a reason”,I have come to a conclusion that you can never...,"{'username': 'gaurav797', 'num_cities': 28, 'n...",October 2012,89585,0,2012-12-20,147767186,False,5.0,5.0,5.0,4.0,5.0,5.0,5.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632474,“Pan Pacific - WOW!”,"The Pan Pacific Hotel, situated in Union Squar...","{'username': '', 'id': '', 'location': ''}",NaN,81444,2,2001-04-25,236069,False,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN
582362,“Accommodation & email”,Motel was clean. Had everything I needed. Lots...,"{'username': '', 'id': '', 'location': ''}",NaN,112289,5,2001-04-13,235211,False,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
789223,“The Presidential Suite”,Our stay at the Fairmont Copley in Boston was ...,"{'username': '', 'id': '', 'location': ''}",NaN,111444,7,2001-04-13,235210,False,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN
761937,“Best Western Boston”,"Excellent accommodations, very considerate in ...","{'username': '', 'id': '', 'location': ''}",NaN,77638,29,2001-03-28,234261,False,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN


# Create authors table and add author key to table

We also see that the author column is nested and contains various information about the author. In this case, we form a separate table dedicated to author data to reduce redundancy (1 author can write multiple reviews).

In [4]:
# Build the author dataframe from the nested author column in the raw dataframe.

author_df = build_authors(sorted_review_df)
author_df.head()

,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
0,netsurfe,3.0,3.0,3.0,4964C44FF14B1AF51602C492E6136B49,Medan,NaN
1,janiejake,10.0,14.0,5.0,1DE75FA343DB9DDE99B70089937968AD,"Blaine, MN",5.0
2,P T,8.0,9.0,5.0,954FDF77404490ED46B4CA9A836782A6,"New York City, New York",11.0
3,INNALOO,17.0,21.0,16.0,BD0F6A7A12AB624C732D3E41415CF9F3,"Singapore, Singapore",19.0
4,gaurav797,28.0,44.0,44.0,B066B484A0FBF6C4D2A596967A671552,abu dhabi,20.0


In [5]:
# Continue author dataframe processing

# Remove duplicates
author_df.drop_duplicates(inplace=True)
author_df

,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
0,netsurfe,3.0,3.0,3.0,4964C44FF14B1AF51602C492E6136B49,Medan,NaN
1,janiejake,10.0,14.0,5.0,1DE75FA343DB9DDE99B70089937968AD,"Blaine, MN",5.0
2,P T,8.0,9.0,5.0,954FDF77404490ED46B4CA9A836782A6,"New York City, New York",11.0
3,INNALOO,17.0,21.0,16.0,BD0F6A7A12AB624C732D3E41415CF9F3,"Singapore, Singapore",19.0
4,gaurav797,28.0,44.0,44.0,B066B484A0FBF6C4D2A596967A671552,abu dhabi,20.0
...,...,...,...,...,...,...,...
878136,CDN_ROCKER,2.0,2.0,NaN,E01681A50DC19FFA18D84EC84718D867,"port huron, MI",1.0
878140,imaginetr,5.0,5.0,5.0,75EACF34E876B736647E8EAE0D9318B7,"Tacoma, Wa",8.0
878142,mzani,47.0,70.0,44.0,35FBC6EA978B4AE51536F1150F32C6AC,"Marseille, France",104.0
878143,a110,2.0,2.0,NaN,8C17DDA718DC7815A1B9ADDD9592EED3,Spain,4.0


In [6]:
# Show the number of unique author_id for each values in the author_id column
author_id_counts = author_df['id'].value_counts()
print(author_id_counts.head(5))

# Show the data entries that have the same author_id
duplicate_author_id_entries = author_df[author_df['id'].isin(author_id_counts[author_id_counts > 1].index)]
duplicate_author_id_entries.sort_values(by='id', inplace=True)
print("Entries with same author_id:")
display(duplicate_author_id_entries.head(5))

# Show the data entries that have the same author_id but different usernames
multi_usernames = author_df[
    author_df.groupby('id')['username'].transform('nunique') > 1
].sort_values('id')
print("Entries with same author_id but different usernames:")
display(multi_usernames.head(10))

# Show data entries with CATID_ as author_id
catid_entries = author_df[author_df['id'] == "CATID_"]
print("Entries with CATID_ as author_id:")
display(catid_entries.head(5))

# Show data entries with empty author_id
empty_id_entries = author_df[author_df['id'] == ""]
print("Entries with empty author_id:")
display(empty_id_entries.head(5))

id
CATID_                              227
4D8C64C57F6DEE4120094521477D5C99      4
0C59F7007DA5F377E6B0AC058A1D81CA      3
F8740706F5BD9058E5056415FD421005      3
E119DF12E80F07933DD52C9DD01D511F      3
Name: count, dtype: int64
Entries with same author_id:


/var/folders/vm/v_v0kznn2hvd1ybg7fsl1yzm0000gn/T/ipykernel_2361/3344215622.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicate_author_id_entries.sort_values(by='id', inplace=True)


,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
158198,Tenley,12.0,28.0,10.0,000F77670B2244A4A131ED2CCFAF1414,"washington, dc",21.0
559677,Tenley,12.0,28.0,10.0,000F77670B2244A4A131ED2CCFAF1414,"washington, dc",20.0
287725,Honu_Ohana,72.0,156.0,42.0,0042497C37852E3FF33EE4162E6FC6BB,"Washington DC, District of Columbia",466.0
293885,Honu_Ohana,72.0,156.0,42.0,0042497C37852E3FF33EE4162E6FC6BB,"Washington DC, District of Columbia",465.0
612230,das_ubersoldat,44.0,88.0,49.0,00BBAC3339F576E164FF9F627489481C,"Hamburg, Germany",35.0


Entries with same author_id but different usernames:


,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
381659,alessandraai,35.0,78.0,28.0,0C59F7007DA5F377E6B0AC058A1D81CA,"Rome, Italy",43.0
99079,Alessandra I,35.0,79.0,28.0,0C59F7007DA5F377E6B0AC058A1D81CA,"Rome, Italy",43.0
99611,alessandraai,35.0,79.0,28.0,0C59F7007DA5F377E6B0AC058A1D81CA,"Rome, Italy",43.0
720393,TravelsNC,7.0,13.0,3.0,18D1A20641D1F3DE0175C19F986082D0,"Mooresville, NC",2.0
237326,giggle731,7.0,13.0,3.0,18D1A20641D1F3DE0175C19F986082D0,"Mooresville, NC",2.0
238282,Bellwmu,14.0,23.0,12.0,1B27A5219983ACC4A0269FAF6F7E20C5,"Ann Arbor, MI",14.0
578057,niki16,14.0,23.0,12.0,1B27A5219983ACC4A0269FAF6F7E20C5,"Ann Arbor, MI",14.0
593032,SueKA,4.0,7.0,6.0,CATID_,US,11.0
669441,RicardoJoana,NaN,1.0,NaN,CATID_,PORTUGAL,NaN
671094,MissTravel11,5.0,5.0,5.0,CATID_,"Oslo, Norway",5.0


Entries with CATID_ as author_id:


,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
35138,Goatlady8,4.0,10.0,NaN,CATID_,,9.0
80300,Greg W,3.0,4.0,3.0,CATID_,"Manhattan, New York, United States",2.0
104378,Daniela P,NaN,1.0,NaN,CATID_,"Los Angeles, California",NaN
136853,love2worknplay,20.0,46.0,12.0,CATID_,Sonora,21.0
148222,Marykeatingeast,2.0,2.0,NaN,CATID_,"Beaumont, Texas",1.0


Entries with empty author_id:


,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
328,,NaN,NaN,NaN,,,NaN


Here we can see that there are authors data entries which contain the same id. Some are have updated numbers of votes and/or number of reviews. Others have multiple usernames. Some also have CATID_ as the id. We will process these cases by selecting the latest entries, collecting any username changes into aliases, and removing users with CATID_.

In [7]:
# Drop author where id equals to "CATID_"
author_df = author_df[author_df['id'] != "CATID_"]

# Drop author where id equals to ""
author_df = author_df[author_df['id'] != ""]

In [8]:
author_df

,username,num_cities,num_reviews,num_type_reviews,id,location,num_helpful_votes
0,netsurfe,3.0,3.0,3.0,4964C44FF14B1AF51602C492E6136B49,Medan,NaN
1,janiejake,10.0,14.0,5.0,1DE75FA343DB9DDE99B70089937968AD,"Blaine, MN",5.0
2,P T,8.0,9.0,5.0,954FDF77404490ED46B4CA9A836782A6,"New York City, New York",11.0
3,INNALOO,17.0,21.0,16.0,BD0F6A7A12AB624C732D3E41415CF9F3,"Singapore, Singapore",19.0
4,gaurav797,28.0,44.0,44.0,B066B484A0FBF6C4D2A596967A671552,abu dhabi,20.0
...,...,...,...,...,...,...,...
878136,CDN_ROCKER,2.0,2.0,NaN,E01681A50DC19FFA18D84EC84718D867,"port huron, MI",1.0
878140,imaginetr,5.0,5.0,5.0,75EACF34E876B736647E8EAE0D9318B7,"Tacoma, Wa",8.0
878142,mzani,47.0,70.0,44.0,35FBC6EA978B4AE51536F1150F32C6AC,"Marseille, France",104.0
878143,a110,2.0,2.0,NaN,8C17DDA718DC7815A1B9ADDD9592EED3,Spain,4.0


In [9]:
# Aggregate author data based on the most recent entry and collect aliases for each author_id (if any)

df = author_df.copy()  

df["most_recent_username"] = df.groupby("id")["username"].transform("first")

agg_df = (
    df.groupby("id", as_index=False)
      .agg(
          username=("username", "first"),
          num_cities=("num_cities", "max"),
          num_helpful_votes=("num_helpful_votes", "max"),
          num_reviews=("num_reviews", "max"),
          num_type_reviews=("num_type_reviews", "max"),
          location=("location", "first"),
      )
)

alias_df = (
    df.loc[df["username"] != df["most_recent_username"], ["id", "username"]]
      .drop_duplicates()
      .groupby("id")["username"]
      .apply(lambda s: ",".join(s.tolist()))
      .reset_index(name="alias")
)

authors_final = agg_df.merge(alias_df, on="id", how="left")
authors_final["alias"] = authors_final["alias"].fillna("")
authors_final.head()

,id,username,num_cities,num_helpful_votes,num_reviews,num_type_reviews,location,alias
0,000024378950A8F32FF775F8E4E8773E,DCJim1,15.0,21.0,22.0,19.0,DCJim1,
1,000025F224EEAC6356F17D0E4356ED2E,chusacj,5.0,4.0,6.0,5.0,Zaragoza,
2,00002DB43A782074B45C6051291F6735,donhme,NaN,4.0,4.0,NaN,Northeast,
3,00003B5E95F3FF95697A63989B822A78,jaris58,NaN,NaN,1.0,NaN,Sao Jose do Rio Preto,
4,0000429353797B21AC67E5C210BC7F20,shrek2000,NaN,NaN,1.0,NaN,St. Louis,


In [10]:
# Alias grouping verification
authors_final[authors_final['id'] == "18D1A20641D1F3DE0175C19F986082D0"]

,id,username,num_cities,num_helpful_votes,num_reviews,num_type_reviews,location,alias
55957,18D1A20641D1F3DE0175C19F986082D0,giggle731,7.0,2.0,13.0,3.0,"Mooresville, NC",TravelsNC


In [11]:
authors_final = authors_final.sort_values(by="num_reviews", ascending=False)

In [12]:
print("\nNumber of unique values per column:")
print(authors_final.nunique())

print("\nDuplicates in authors_final:", authors_final.duplicated().sum())
print(f"\nNumber of unique rows in authors_final: {len(authors_final.drop_duplicates())}")


Number of unique values per column:
id                   576687
username             536732
num_cities              183
num_helpful_votes       501
num_reviews             377
num_type_reviews        190
location              77972
alias                     5
dtype: int64

Duplicates in authors_final: 0

Number of unique rows in authors_final: 576687


# Clean up review table

In [13]:
# Extract author_id and filter to only include valid authors from the author table

sorted_review_df = attach_author_id(sorted_review_df)

valid_author_ids = author_df["id"].dropna().unique()

sorted_review_df = sorted_review_df[
    sorted_review_df["author_id"].isin(valid_author_ids)
].copy()

sorted_review_df.head(5)

,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_id
384302,“Place to stay in Downtown LA”,This is my first time staying in downtown when...,November 2012,77804,0,2012-12-20,147786903,False,4.0,4.0,5.0,5.0,5.0,5.0,4.0,NaN,NaN,4964C44FF14B1AF51602C492E6136B49
266659,“Michigan Avenue/Grant Park”,"Great location, right on Michigan Ave. Bus sto...",August 2012,87573,0,2012-12-20,147767791,False,4.0,3.0,3.0,3.0,4.0,4.0,3.0,NaN,NaN,1DE75FA343DB9DDE99B70089937968AD
840520,“Excellent location”,Location: The location was very good as it was...,August 2012,84137,0,2012-12-20,147777249,False,4.0,4.0,4.0,5.0,5.0,5.0,3.0,NaN,NaN,954FDF77404490ED46B4CA9A836782A6
840518,"“Convenient location, but need a makeover”","First trip to Washington, children want to see...",November 2012,84067,0,2012-12-20,147771855,False,5.0,5.0,4.0,4.0,5.0,3.0,3.0,NaN,NaN,BD0F6A7A12AB624C732D3E41415CF9F3
781827,“four seasons for a reason”,I have come to a conclusion that you can never...,October 2012,89585,0,2012-12-20,147767186,False,5.0,5.0,5.0,4.0,5.0,5.0,5.0,NaN,NaN,B066B484A0FBF6C4D2A596967A671552


## Remove rows with missing data

In [14]:
# Check missing values in sorted_review_df in percentage
missing_values = sorted_review_df.isnull().sum()
missing_percentage = (missing_values / len(sorted_review_df)) * 100
missing_percentage

title                       0.000000
text                        0.000000
date_stayed                 1.725795
hotel_id                    0.000000
num_helpful_votes           0.000000
review_date                 0.000000
review_id                   0.000000
via_mobile                  0.000000
service_rating              8.195248
cleanliness_rating          8.190630
overall_rating              0.000000
value_rating                8.189881
location_rating            17.602434
sleep_quality_rating       37.969860
rooms_rating               15.128633
check_in_service_rating    87.565340
business_service_rating    91.801756
author_id                   0.000000
dtype: float64

In [15]:
# Check missing values in sorted_review_df based on years

df = sorted_review_df.copy()

df["year"] = df["review_date"].dt.year

missing_by_year = (
    df
    .groupby("year")
    .apply(lambda x: x.isna().mean() * 100)
)

print("Percentage of missing values by year:")
missing_by_year.round(2)

Percentage of missing values by year:


/var/folders/vm/v_v0kznn2hvd1ybg7fsl1yzm0000gn/T/ipykernel_2361/3390506466.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isna().mean() * 100)


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_id,year
year,,,,,,,,,,,,,,,,,,,
2002,0.0,0.0,84.96,0.0,0.0,0.0,0.0,0.0,84.96,84.96,0.0,84.96,100.00,100.00,84.96,100.00,100.00,0.0,0.0
2003,0.0,0.0,86.05,0.0,0.0,0.0,0.0,0.0,86.26,86.31,0.0,86.36,100.00,100.00,86.15,100.00,100.00,0.0,0.0
2004,0.0,0.0,73.16,0.0,0.0,0.0,0.0,0.0,73.39,73.27,0.0,73.36,100.00,100.00,73.19,100.00,100.00,0.0,0.0
2005,0.0,0.0,19.71,0.0,0.0,0.0,0.0,0.0,25.76,25.38,0.0,25.71,100.00,100.00,25.38,100.00,100.00,0.0,0.0
2006,0.0,0.0,7.35,0.0,0.0,0.0,0.0,0.0,19.45,17.46,0.0,18.11,52.97,100.00,17.42,52.89,70.65,0.0,0.0
2007,0.0,0.0,0.91,0.0,0.0,0.0,0.0,0.0,19.73,16.20,0.0,17.50,16.03,100.00,16.23,16.01,45.06,0.0,0.0
2008,0.0,0.0,0.68,0.0,0.0,0.0,0.0,0.0,14.14,10.04,0.0,11.00,9.86,100.00,10.09,9.84,39.93,0.0,0.0
2009,0.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,6.97,6.48,0.0,7.14,6.52,100.00,6.85,92.58,94.73,0.0,0.0
2010,0.0,0.0,0.17,0.0,0.0,0.0,0.0,0.0,6.19,6.05,0.0,6.67,6.14,14.23,6.38,99.95,99.96,0.0,0.0


Observations:
- Sleep quality rating was only introduced in 2010, therefore all reviews from 2008 and 2009 don't have sleep_quality_rating
- Check in service rating and business service rating may be dropped since more than 90% of the data is missing from the recent 5 years

In [16]:
ratings_to_require = [
    "date_stayed",
    "service_rating",
    "cleanliness_rating",
    "overall_rating",
    "value_rating",
    "location_rating",
    "rooms_rating"
]

cleaned_review_df = drop_missing_reviews(sorted_review_df, ratings_to_require)

## Filter to include recent years and relevant reviews

In [17]:
# Filter data to only include reviews from the most recent 5 years

cleaned_review_df = cleaned_review_df.rename(columns={'date': 'review_date'})
latest_year = cleaned_review_df['review_date'].max().year

filtered_review_df = cleaned_review_df[cleaned_review_df['review_date'].dt.year >= latest_year - 4]
print(f"Date range of reviews: {filtered_review_df['review_date'].min()} to {filtered_review_df['review_date'].max()}")


Date range of reviews: 2008-01-01 00:00:00 to 2012-12-20 00:00:00


In [18]:
filtered_review_df = filtered_review_df.copy()

filtered_review_df["review_date"] = pd.to_datetime(filtered_review_df["review_date"], errors="coerce")

filtered_review_df["date_stayed"] = pd.to_datetime(filtered_review_df["date_stayed"], format="%B %Y", errors="coerce")
filtered_review_df["approx_stay_date"] = filtered_review_df["date_stayed"] + pd.offsets.MonthEnd(0)

review_month = filtered_review_df["review_date"].dt.to_period("M")
stay_month = filtered_review_df["approx_stay_date"].dt.to_period("M")

filtered_review_df["month_diff"] = (review_month - stay_month).apply(
    lambda x: x.n if pd.notna(x) else None
)

filtered_review_df = filtered_review_df[
    filtered_review_df["month_diff"].notna() & (filtered_review_df["month_diff"] >= 0)
]

filtered_review_df.head(5)

,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_id,approx_stay_date,month_diff
384302,“Place to stay in Downtown LA”,This is my first time staying in downtown when...,2012-11-01,77804,0,2012-12-20,147786903,False,4.0,4.0,5.0,5.0,5.0,5.0,4.0,NaN,NaN,4964C44FF14B1AF51602C492E6136B49,2012-11-30,1.0
266659,“Michigan Avenue/Grant Park”,"Great location, right on Michigan Ave. Bus sto...",2012-08-01,87573,0,2012-12-20,147767791,False,4.0,3.0,3.0,3.0,4.0,4.0,3.0,NaN,NaN,1DE75FA343DB9DDE99B70089937968AD,2012-08-31,4.0
840520,“Excellent location”,Location: The location was very good as it was...,2012-08-01,84137,0,2012-12-20,147777249,False,4.0,4.0,4.0,5.0,5.0,5.0,3.0,NaN,NaN,954FDF77404490ED46B4CA9A836782A6,2012-08-31,4.0
840518,"“Convenient location, but need a makeover”","First trip to Washington, children want to see...",2012-11-01,84067,0,2012-12-20,147771855,False,5.0,5.0,4.0,4.0,5.0,3.0,3.0,NaN,NaN,BD0F6A7A12AB624C732D3E41415CF9F3,2012-11-30,1.0
781827,“four seasons for a reason”,I have come to a conclusion that you can never...,2012-10-01,89585,0,2012-12-20,147767186,False,5.0,5.0,5.0,4.0,5.0,5.0,5.0,NaN,NaN,B066B484A0FBF6C4D2A596967A671552,2012-10-31,2.0


In [19]:
total = len(filtered_review_df)

less_1 = (filtered_review_df["month_diff"] <= 1).sum()
less_2 = (filtered_review_df["month_diff"] <= 2).sum()
less_3 = (filtered_review_df["month_diff"] <= 3).sum()

print("Percentage of reviews the gap between stay_date and review_date is within:")
print(f"≤ 1 month: {less_1} ({pct(less_1,total)})")
print(f"≤ 2 months: {less_2} ({pct(less_2,total)})")
print(f"≤ 3 months: {less_3} ({pct(less_3,total)})")

Percentage of reviews the gap between stay_date and review_date is within:
≤ 1 month: 503198 (84.75%)
≤ 2 months: 528219 (88.97%)
≤ 3 months: 540861 (91.10%)


Seeing that we still have more than 80% of data if we filter the reviews to only include reviews done at max 1 month after the stay, we can remove the irrelevant data.

In [20]:
filtered_review_df = filtered_review_df[filtered_review_df['month_diff'] <= 1]
filtered_review_df.drop(columns=['month_diff', 'approx_stay_date'], inplace=True)
filtered_review_df.head(5)

,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_id
384302,“Place to stay in Downtown LA”,This is my first time staying in downtown when...,2012-11-01,77804,0,2012-12-20,147786903,False,4.0,4.0,5.0,5.0,5.0,5.0,4.0,NaN,NaN,4964C44FF14B1AF51602C492E6136B49
840518,"“Convenient location, but need a makeover”","First trip to Washington, children want to see...",2012-11-01,84067,0,2012-12-20,147771855,False,5.0,5.0,4.0,4.0,5.0,3.0,3.0,NaN,NaN,BD0F6A7A12AB624C732D3E41415CF9F3
865265,“Unbelievable customer service”,We arrived late at night and immediately were ...,2012-12-01,120556,0,2012-12-20,147785077,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,631AAF84885D8AA48F4876100F92CEEB
756612,“Satisfactory Accomodations with Satisfactory ...,This hotel is conveniently located both to Log...,2012-11-01,268205,0,2012-12-20,147768078,False,5.0,5.0,4.0,5.0,5.0,5.0,5.0,NaN,NaN,4D4F5E731C6F36026CE4CE0E987167C7
381763,“Best location”,Great location just close to all LA areas....a...,2012-12-01,1439222,0,2012-12-20,147788784,False,4.0,4.0,4.0,4.0,5.0,NaN,4.0,NaN,NaN,94F1930119DE5551490416C7DED363CE


## Check for duplicates

In [21]:
n = len(filtered_review_df)

dup_cols = [c for c in ["title","text"] if c in filtered_review_df.columns]
if dup_cols:
    dup_cnt = filtered_review_df.duplicated(subset=dup_cols).sum()
    print("Potential duplicate content rows:", dup_cnt, pct(dup_cnt, n))

duplicate_rows = filtered_review_df[
    filtered_review_df.duplicated(subset=dup_cols, keep=False)
].sort_values(dup_cols)

print("Number of rows involved in potential duplicates:", len(duplicate_rows))
duplicate_rows.head(10)

Potential duplicate content rows: 47 0.01%
Number of rows involved in potential duplicates: 94


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_id
177746,“5 star service - comfy rooms!”,"Room service was great, so was the entire bell...",2011-04-01,1646128,0,2011-04-15,104169141,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,4D2533FE80EA1B398A35330AADAD9F17
177747,“5 star service - comfy rooms!”,"Room service was great, so was the entire bell...",2011-04-01,1646128,0,2011-04-15,104146507,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,4D2533FE80EA1B398A35330AADAD9F17
695428,“Awesome Hotel”,After a long day on the road this was the perf...,2011-11-01,1409115,1,2011-11-01,120028598,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,B2D8508D72D259A1A76133C87AEDF2F5
697850,“Awesome Hotel”,After a long day on the road this was the perf...,2009-04-01,1409115,1,2009-04-14,28040444,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,836EA6851C1DCB13B0A2307AE00B7166
454278,“Awesomeness”,Ashley was amazing at service at the front des...,2011-07-01,258655,0,2011-08-28,117340462,True,4.0,5.0,5.0,4.0,5.0,NaN,5.0,NaN,NaN,A579881C7B0584109ABC1F0F139EA3C2
454304,“Awesomeness”,Ashley was amazing at service at the front des...,2011-07-01,258655,0,2011-08-28,117337091,True,4.0,5.0,5.0,4.0,5.0,NaN,5.0,NaN,NaN,A579881C7B0584109ABC1F0F139EA3C2
759975,“Beautiful Neighbourhood Hotel”,Lovely updated hotel in beautiful Boston Back ...,2012-06-01,89568,0,2012-06-14,131956851,False,5.0,5.0,4.0,4.0,5.0,5.0,4.0,NaN,NaN,AE8917FED3CE1E039B059EEA08901210
759927,“Beautiful Neighbourhood Hotel”,Lovely updated hotel in beautiful Boston Back ...,2012-06-01,89568,0,2012-06-14,131957252,False,5.0,5.0,4.0,4.0,5.0,5.0,4.0,NaN,NaN,AE8917FED3CE1E039B059EEA08901210
488591,"“Beautiful Old World style, furnishings, and s...",We adored our stay here. We got our room at a ...,2011-07-01,99458,0,2011-08-17,116832160,True,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,303F66EA987C71414841F13559F013C2
488562,"“Beautiful Old World style, furnishings, and s...",We adored our stay here. We got our room at a ...,2011-07-01,99458,0,2011-08-17,116832161,True,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,303F66EA987C71414841F13559F013C2


In [22]:
# Sort so the earliest date comes first
review_df, sorted_filtered_review_df = dedupe_reviews_by_content(
    filtered_review_df,
    subset=['hotel_id', 'title', 'text'],
    date_col='review_date'
)

print('Before:', sorted_filtered_review_df.shape)
print('After :', review_df.shape)
print('Removed:', len(filtered_review_df) - len(review_df))

Before: (503198, 18)
After : (503160, 18)
Removed: 38


# Create hotel table

In [23]:
rating_cols = ["service_rating", "cleanliness_rating", "overall_rating", "value_rating", "location_rating", "sleep_quality_rating", "rooms_rating", "check_in_service_rating", "business_service_rating"]

hotels_df = build_hotels_agg(review_df, rating_cols)

hotels_df

,hotel_id,num_reviews,avg_service_rating,avg_cleanliness_rating,avg_overall_rating,avg_value_rating,avg_location_rating,avg_sleep_quality_rating,avg_rooms_rating,avg_check_in_service_rating,avg_business_service_rating
0,214197,1825,2.363836,2.306301,2.261370,2.617534,4.433425,2.595794,2.065205,2.421053,2.265306
1,122005,1774,3.911499,4.221533,3.949267,3.858512,4.665163,4.098581,3.717587,3.623377,3.843137
2,93520,1739,3.522714,3.600920,3.550316,3.557217,4.687752,3.813973,3.456009,2.994012,2.568182
3,93562,1685,4.252819,4.217804,4.077151,4.016617,4.670623,3.936960,3.997033,4.369369,3.757576
4,93618,1647,3.993321,4.101396,3.828780,3.408622,4.602914,4.099425,3.680631,4.134328,3.474359
...,...,...,...,...,...,...,...,...,...,...,...
3787,109277,1,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN
3788,488773,1,5.000000,5.000000,5.000000,5.000000,5.000000,NaN,5.000000,NaN,NaN
3789,108974,1,4.000000,4.000000,3.000000,4.000000,4.000000,NaN,4.000000,NaN,NaN
3790,108935,1,4.000000,4.000000,3.000000,3.000000,3.000000,NaN,4.000000,NaN,NaN


# Final checks before transforming into SQLite

In [24]:
print("\n=== DATA FOUNDATION SUMMARY ===")
print("Total reviews:", len(review_df))
print("Unique hotels:", review_df["hotel_id"].nunique())
if "author_key" in review_df.columns:
    print("Unique authors in reviews:", review_df["author_key"].nunique())
print("Review date range:", review_df["review_date"].min(), "→", review_df["review_date"].max())
core = [c for c in ["overall_rating","service_rating","cleanliness_rating","value_rating","location_rating","rooms_rating"] if c in review_df.columns]
if core:
    kept = len(review_df.dropna(subset=core))
    print(f"Rows with all core ratings present ({len(core)} cols):", kept, pct(kept, len(review_df)))


=== DATA FOUNDATION SUMMARY ===
Total reviews: 503160
Unique hotels: 3792
Review date range: 2008-01-01 00:00:00 → 2012-12-20 00:00:00
Rows with all core ratings present (6 cols): 503160 100.00%


In [25]:
review_cols = ["hotel_id", "review_id"]
review_df = review_df.copy()
for col in review_cols:
    review_df[col] = review_df[col].astype("string")

review_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 503160 entries, 465859 to 384302
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   title                    503160 non-null  object        
 1   text                     503160 non-null  object        
 2   date_stayed              503160 non-null  datetime64[ns]
 3   hotel_id                 503160 non-null  string        
 4   num_helpful_votes        503160 non-null  int64         
 5   review_date              503160 non-null  datetime64[ns]
 6   review_id                503160 non-null  string        
 7   via_mobile               503160 non-null  bool          
 8   service_rating           503160 non-null  float64       
 9   cleanliness_rating       503160 non-null  float64       
 10  overall_rating           503160 non-null  float64       
 11  value_rating             503160 non-null  float64       
 12  location_rating 

In [26]:
authors_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 576687 entries, 184361 to 560315
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 576687 non-null  object 
 1   username           576687 non-null  object 
 2   num_cities         394266 non-null  float64
 3   num_helpful_votes  436231 non-null  float64
 4   num_reviews        576634 non-null  float64
 5   num_type_reviews   300525 non-null  float64
 6   location           576687 non-null  object 
 7   alias              576687 non-null  object 
dtypes: float64(4), object(4)
memory usage: 39.6+ MB


In [27]:
hotels_df["hotel_id"] = hotels_df["hotel_id"].astype("string")

hotels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3792 entries, 0 to 3791
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   hotel_id                     3792 non-null   string 
 1   num_reviews                  3792 non-null   int64  
 2   avg_service_rating           3792 non-null   float64
 3   avg_cleanliness_rating       3792 non-null   float64
 4   avg_overall_rating           3792 non-null   float64
 5   avg_value_rating             3792 non-null   float64
 6   avg_location_rating          3792 non-null   float64
 7   avg_sleep_quality_rating     3686 non-null   float64
 8   avg_rooms_rating             3792 non-null   float64
 9   avg_check_in_service_rating  2625 non-null   float64
 10  avg_business_service_rating  2485 non-null   float64
dtypes: float64(9), int64(1), string(1)
memory usage: 326.0 KB


# Transform into sqlite (full data and sample)

In [28]:
analysis_df, sample_df = make_reproducible_samples(
    review_df,
    analysis_n=80000,
    sample_n=6000,
    seed=42,
)

# Pre-compute variance from each subset so the hotels table stores values
# consistent with the reviews in each DB (eliminates runtime groupby in NB02).
# Drop any stale variance columns first so the merge is always clean.
def _subset_variance(subset, cols, id_col='hotel_id'):
    return (
        subset[[id_col] + cols]
        .groupby(id_col)[cols]
        .var(ddof=0)
        .rename(columns={c: f'{c}_variance' for c in cols})
        .reset_index()
    )

hotels_base = hotels_df.drop(
    columns=[c for c in hotels_df.columns if c.endswith('_variance')],
    errors='ignore'
)

hotels_analysis_df = hotels_base.merge(_subset_variance(analysis_df, rating_cols), on='hotel_id', how='left')
hotels_sample_df   = hotels_base.merge(_subset_variance(sample_df,   rating_cols), on='hotel_id', how='left')

write_db("../data/reviews_analysis.db", analysis_df, authors_final, hotels_analysis_df)
write_db("../data/reviews_sample.db",   sample_df,   authors_final, hotels_sample_df)

# Check DB file

In [29]:
import sqlite3

conn = sqlite3.connect("../data/reviews_analysis.db")

pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

,name
0,reviews
1,authors
2,hotels


In [30]:
# Number of records per table
print(pd.read_sql("SELECT COUNT(*) AS n_reviews FROM reviews", conn))
print(pd.read_sql("SELECT COUNT(*) AS n_authors FROM authors", conn))
print(pd.read_sql("SELECT COUNT(*) AS n_hotels  FROM hotels",  conn))

   n_reviews
0      80000
   n_authors
0      75349
   n_hotels
0      3380


In [31]:
# Describe data
print(pd.read_sql("PRAGMA table_info(reviews)", conn))
print(pd.read_sql("PRAGMA table_info(authors)", conn))
print(pd.read_sql("PRAGMA table_info(hotels)", conn))

    cid                     name       type  notnull dflt_value  pk
0     0                    title       TEXT        0       None   0
1     1                     text       TEXT        0       None   0
2     2              date_stayed  TIMESTAMP        0       None   0
3     3                 hotel_id       TEXT        0       None   0
4     4        num_helpful_votes    INTEGER        0       None   0
5     5              review_date  TIMESTAMP        0       None   0
6     6                review_id       TEXT        0       None   0
7     7               via_mobile    INTEGER        0       None   0
8     8           service_rating       REAL        0       None   0
9     9       cleanliness_rating       REAL        0       None   0
10   10           overall_rating       REAL        0       None   0
11   11             value_rating       REAL        0       None   0
12   12          location_rating       REAL        0       None   0
13   13     sleep_quality_rating       REAL     

In [32]:
# Display indexes
pd.read_sql(
    "SELECT name, tbl_name FROM sqlite_master WHERE type='index'",
    conn
)

,name,tbl_name
0,idx_reviews_hotel,reviews
1,idx_reviews_date,reviews
2,idx_reviews_hotel_date,reviews
3,ux_reviews_review_id,reviews
4,ux_hotels_hotel_id,hotels
5,idx_authors_id,authors
6,ux_authors_id,authors
7,idx_hotels_hotel_id,hotels


In [33]:
# Check number of duplicates in reviews
pd.read_sql("""
SELECT COUNT(*) AS dup_review_ids
FROM (
    SELECT review_id
    FROM reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
)
""", conn)


,dup_review_ids
0,0


In [34]:
# Check number of orphan reviews
pd.read_sql("""
SELECT COUNT(*) AS orphan_reviews
FROM reviews r
LEFT JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE h.hotel_id IS NULL
""", conn)


,orphan_reviews
0,0


In [35]:
# Number of reviews of which the hotel does not exist in the hotels database
pd.read_sql("""
SELECT COUNT(*) 
FROM reviews r
LEFT JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE h.hotel_id IS NULL;""", conn)

,COUNT(*)
0,0


In [36]:
# Number of reviews of which the author does not exist in the author database
pd.read_sql("""
SELECT COUNT(*)
FROM reviews r
LEFT JOIN authors a ON r.author_id = a.id
WHERE a.id IS NULL;""", conn)

,COUNT(*)
0,0


In [37]:
# Show overall ratings for hotel
pd.read_sql("""
SELECT review_id, hotel_id, review_date, overall_rating
FROM reviews
LIMIT 10
""", conn)


,review_id,hotel_id,review_date,overall_rating
0,145753174,225108,2012-11-19 00:00:00,5.0
1,81870377,122005,2010-10-02 00:00:00,4.0
2,33732815,939375,2009-07-02 00:00:00,4.0
3,32773886,87573,2009-06-21 00:00:00,4.0
4,38157829,123022,2009-08-19 00:00:00,4.0
5,124554278,81803,2012-02-13 00:00:00,4.0
6,147340920,81520,2012-12-12 00:00:00,4.0
7,110145670,223024,2011-05-27 00:00:00,4.0
8,122462692,75688,2012-01-03 00:00:00,4.0
9,135363818,112412,2012-07-25 00:00:00,4.0


In [38]:
# Check that composite indexes are used
pd.read_sql("""
EXPLAIN QUERY PLAN
SELECT AVG(overall_rating)
FROM reviews
WHERE hotel_id = (
    SELECT hotel_id FROM reviews LIMIT 1
)
AND review_date >= '2011-01-01'
""", conn)


,id,parent,notused,detail
0,4,0,0,SEARCH reviews USING INDEX idx_reviews_hotel_d...
1,7,0,0,SCALAR SUBQUERY 1
2,15,7,0,SCAN reviews USING COVERING INDEX idx_reviews_...


In [39]:
conn.close()